# RAPIDS Visualization Guide Successful Reference

A compact reference implementation for the portable GPU visualization skill. This notebook demonstrates maintained patterns for a Divvy-style dataset without relying on cuxfilter. It is stored under `examples/` as a human/golden reference, not as an eval fixture.

## Setup

The notebook keeps CPU/GPU boundaries explicit. Use `BACKEND = "auto"` for cuDF with pandas fallback, `"cudf"` to require GPU, or `"pandas"` for CPU review.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import urllib.request

import holoviews as hv
import panel as pn
import hvplot.pandas  # noqa: F401

try:
    import cudf
    import hvplot.cudf  # noqa: F401
except Exception:
    cudf = None

import pandas as pd

hvplot.extension("bokeh")
pn.extension("tabulator")

In [ ]:
S3 = "https://divvy-tripdata.s3.amazonaws.com/"
DATA_DIR = Path("data")
BACKEND = "auto"
DOWNLOAD_DATA = False
YEARS = range(2021, 2022)
MONTHS = range(1, 13)

In [ ]:
def download_divvy_data(data_dir=DATA_DIR):
    data_dir.mkdir(parents=True, exist_ok=True)
    for year in YEARS:
        for month in MONTHS:
            file_name = f"{year}{month:02d}-divvy-tripdata.zip"
            zip_path = data_dir / file_name
            if not zip_path.exists():
                urllib.request.urlretrieve(f"{S3}{file_name}", zip_path)
            with ZipFile(zip_path) as zip_file:
                zip_file.extractall(data_dir)

if DOWNLOAD_DATA:
    download_divvy_data()

## Load and Clean

In [ ]:
def read_csv(path, *, backend=BACKEND):
    if backend not in {"auto", "cudf", "pandas"}:
        raise ValueError("backend must be auto, cudf, or pandas")

    if backend in {"auto", "cudf"} and cudf is not None:
        try:
            return cudf.read_csv(path), "cudf"
        except Exception:
            if backend == "cudf":
                raise

    return pd.read_csv(path), "pandas"

def concat_frames(frames, backend):
    if backend == "cudf":
        return cudf.concat(frames)
    return pd.concat(frames, ignore_index=True)

def load_trip_data(data_dir=DATA_DIR, backend=BACKEND):
    frames = []
    selected_backend = None
    for file_path in sorted(Path(data_dir).rglob("20*.csv")):
        frame, selected_backend = read_csv(file_path, backend=backend)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError("No Divvy CSV files found. Set DOWNLOAD_DATA=True or provide local CSV files.")
    return concat_frames(frames, selected_backend), selected_backend

def to_pandas(df):
    return df.to_pandas() if hasattr(df, "to_pandas") else df

In [ ]:
def clean_trip_data(df, backend):
    df = df.dropna(subset=["end_lat"])
    df["start_station_name"] = df["start_station_name"].fillna("none")
    df["end_station_name"] = df["end_station_name"].fillna("none")

    df = df[
        (df["start_lat"] >= 41.5) & (df["start_lat"] <= 42.5)
        & (df["start_lng"] >= -88.0) & (df["start_lng"] <= -87.0)
        & (df["end_lat"] >= 41.5) & (df["end_lat"] <= 42.5)
        & (df["end_lng"] >= -88.0) & (df["end_lng"] <= -87.0)
    ]

    if backend == "cudf":
        df["started_at"] = cudf.to_datetime(df["started_at"])
        df["ended_at"] = cudf.to_datetime(df["ended_at"])
    else:
        df["started_at"] = pd.to_datetime(df["started_at"])
        df["ended_at"] = pd.to_datetime(df["ended_at"])

    df["year"] = df["started_at"].dt.year
    df["month"] = df["started_at"].dt.month
    df["day"] = df["started_at"].dt.day
    df["hour"] = df["started_at"].dt.hour
    df["day_of_week"] = df["started_at"].dt.dayofweek
    df["dur_min"] = ((df["ended_at"] - df["started_at"]).dt.seconds / 60).round().astype("float32")

    return df.drop(
        ["ride_id", "started_at", "ended_at", "start_station_id", "end_station_id"],
        axis=1,
    ).reset_index(drop=True)

# df, backend = load_trip_data()
# df = clean_trip_data(df, backend)

## Chart Helpers

In [ ]:
def top_n_categories(df, column, n=20):
    counts = df.groupby(column).size().reset_index(name="count")
    return counts.sort_values("count", ascending=False).head(n)

def unique_values(df, column, *, limit=None):
    values = df[column].dropna().unique()
    if hasattr(values, "to_arrow"):
        out = values.to_arrow().to_pylist()
    elif hasattr(values, "to_list"):
        out = values.to_list()
    elif hasattr(values, "tolist"):
        out = values.tolist()
    else:
        out = list(values)
    out = sorted(out)
    return out[:limit] if limit else out

## Basic hvPlot Views

In [ ]:
ride_type_bars = top_n_categories(df, "rideable_type").hvplot.bar(
    x="rideable_type", y="count", title="Trips by bike type", responsive=True
)

duration_hist = df.hvplot.hist(
    y="dur_min", bins=60, title="Trip duration distribution", responsive=True
)

hourly = df.groupby(["day_of_week", "hour"]).size().reset_index(name="count")
hour_heatmap = hourly.hvplot.heatmap(
    x="hour", y="day_of_week", C="count", cmap="viridis", responsive=True
)

ride_type_bars + duration_hist + hour_heatmap

## Dense Spatial View with Datashader

In [ ]:
start_density = df.hvplot.hexbin(
    x="start_lng",
    y="start_lat",
    geo=True,
    tiles=True,
    gridsize=80,
    cmap="viridis",
    title="Trip start density",
    responsive=True,
)

end_density = df.hvplot.hexbin(
    x="end_lng",
    y="end_lat",
    geo=True,
    tiles=True,
    gridsize=80,
    cmap="magma",
    title="Trip end density",
    responsive=True,
)

start_density + end_density

## Linked Panel Dashboard

In [ ]:
linker = hv.link_selections.instance()

scatter = df.hvplot.scatter(
    x="start_lng",
    y="start_lat",
    c="dur_min",
    rasterize=True,
    cnorm="eq_hist",
    cmap="viridis",
    title="Trip starts colored by duration",
    responsive=True,
    height=420,
)

month_bars = top_n_categories(df, "month", n=12).hvplot.bar(
    x="month", y="count", title="Trips by month", responsive=True
)

hour_hist = df.hvplot.hist(y="hour", bins=24, title="Trips by hour", responsive=True)
linked_view = linker((scatter + month_bars + hour_hist).cols(2))

backend_indicator = pn.widgets.StaticText(name="Data backend", value=backend)
row_count = pn.indicators.Number(name="Rows loaded", value=len(df), format="{value:,}")
sample_table = pn.widgets.Tabulator(to_pandas(df.head(1000)), pagination="remote", page_size=25, height=320)

dashboard = pn.template.FastListTemplate(
    title="RAPIDS Divvy Visualization Guide",
    sidebar=[backend_indicator, row_count],
    main=[linked_view, pn.pane.Markdown("### Sample rows"), sample_table],
)

dashboard.servable()
dashboard

## Run Notes

- Execute load/clean cells before visualization cells.
- Use `BACKEND = "pandas"` for CPU-only review.
- Use `BACKEND = "cudf"` when you want GPU execution to fail loudly if cuDF is unavailable.
- Tables are intentionally bounded before conversion/display.